# Bit-Banging 1-Wire with `scope.bitbanger`
`scope.bitbanger` can be made to speak many different protocols. In this example, we'll have it speak the [1-wire protocol](https://en.wikipedia.org/wiki/1-Wire).

1-wire targets are quite interesting to work with because they typically have only two pins: ground and I/O.

There is no supply voltage, and there is no clock. This presents a fun challenge for side-channel analysis and fault attacks! In this notebook, we'll show:
1. How to use `scope.bitbanger` to "talk" 1-wire.
2. What information about the target's hidden inner workings can still be obtained, despite not having access to its clock and internal power supply.

The 1-wire protocol was designed by Dallas Semiconductor, which was then acquired by Maxim, then Analog Devices.

If you have a 1-wire device from one of these manufacturers, then it's quite likely that our [1-wire helper class](https://chipwhisperer.readthedocs.io/en/latest/scope-api.html#chipwhisperer.capture.scopes.cwhardware.ChipWhispererHuskyBitBanger.OneWireHelper) will prove useful.

Other manufacturers have similar protocols - for example, Atmel/Microchip's "single-wire". For these you'll likely need to study datasheets and write your own transaction methods.

This notebook is written specifically for Analog Devices' DS2431 EEPROM; it may also work on other EEPROMs that use the same protocol.

If you try a different EEPROM, watch out for:
- Differences in the memory map.
- Differences in scratchpad size.

In order to run this notebook, you'll have to connect your 1-wire target to Husky; at this time we don't offer pre-made 1-wire target boards, but it's easy to make your own:

<img src='img/onewirecircuit.png' width=500>

This circuit can easily be implemented on the blank target boards that come with our [starter kits](https://chipwhisperer.readthedocs.io/en/latest/Starter%20Kits/SCAPACK-L1.html), or on a breadboard that is connected to a [CW308](https://chipwhisperer.readthedocs.io/en/latest/Targets/CW308%20UFO.html) or [CW313](https://chipwhisperer.readthedocs.io/en/latest/Targets/CW313.html).

Consult your target's datasheet for recommended pullup resistor values; for the DS2431, the recommended range is 300 - 2.2 k$\ohm$. We've had success with R1 = R2 = 75$\ohm$, which is a bit under spec but results in nicer power traces. We have used C1 = C2 = 100 nF. What matters most is that you get consistent successful communication with your target.

In [ ]:
SCOPE = 'OPENADC'
PLATFORM = 'CWHUSKY'

In [ ]:
%run '/home/jpnewae/git/cw_trimmed/jupyter/Setup_Scripts/Setup_Generic.ipynb'

In [ ]:
scope.bitbanger.onewire.set_defaults()
print(scope.bitbanger)

The above set some useful defaults, but we still need to define which Husky pin is connected to the target's I/O line.

In [ ]:
scope.bitbanger.data_pin = 'USERIO_D0' # or another pin of your choosing
scope.bitbanger.clock_pin = 'disabled' # putting out a clock would add unwanted clock noise to your power traces

We begin by defining a target class that implements the commands required to interact with the DS2431.

Refer to the DS2431 datasheet for all the details, but basically:
- **reading memory** is quite straightforward
- **writing memory** is less so; it requires the following steps:
    - send the destination address and write the data to a scratchpad
    - read back the scratchpad to ensure its integrity and obtain an "authorization code"
    - send back the authorization code with a command to copy the scratchpad contents to the actual memory

In [ ]:
import math
import time

class DS2431(cw.capture.scopes.cwhardware.ChipWhispererHuskyBitBanger.OneWireHelper):
    def __init__(self, bitbanger):
        super().__init__(bitbanger)

    def read_rom_code(self, verbose=False):
        return super().read_rom_code(0x33, expected_family_code=0x2d, verbose=verbose)
        
    def mem_protect_status(self):
        blocks = self.read_mem(0x80, 4) 
        for i,status in enumerate(blocks):
            print('Block %d: ' % i, end='')
            if status == 0x55:
                print('❌ write protected')
            elif status == 0xaa:
                print('❌ EPROM mode')
            elif status == 0x00:
                print('✅ not protected')
            else:
                print('🔥 Unexpected status: 0x%x' % status)
            
        mem_lock = self.read_mem(0x84, 1)[0]
        print('Copy Protection Byte: ', end='')
        if mem_lock in [0x55, 0xaa]:
            print('❌ on')
        elif mem_lock == 0:
            print('✅ off')
        else:
            print('🔥 unexpected value: 0x%x' % mem_lock)
        
        reg_lock = self.read_mem(0x85, 1)[0]
        print('Factory Byte: ', end='')
        if reg_lock in [0x55, 0xaa]:
            print('❌ on')
        elif reg_lock == 0:
            print('✅ off')
        else:
            print('🔥 unexpected value: 0x%x' % reg_lock)

    def read_mem(self, address=0, num_bytes=1, gap=0, trigger_en=True):
        rdata = []
        self.send_rst_pd(trigger_en=False)
        address_bytes = list(int.to_bytes(address, length=2, byteorder='little'))
        read_command = [0xcc, 0xf0]
        read_command.extend(address_bytes)
        self.bb.sendpacket(self.get_generic_write_read(read_command, 0, gap=gap), trigger_en=trigger_en)
        
        burst_size = 4
        reads = math.ceil(num_bytes/burst_size)
          
        for i in range(reads):
            if (num_bytes % burst_size) and (i == reads - 1): 
                size = num_bytes % burst_size
            else:
                size = burst_size
            self.bb.sendpacket(self.get_generic_write_read([], size, gap=0), trigger_en=trigger_en)
            time.sleep(0.1)
            rdata.extend(self.bb.recorded_data(size, False))
        return rdata

    def write_full_scratchpad(self, address, data, verbose=False):
        assert len(data) == 8
        self.send_rst_pd(trigger_en=True)
        address_bytes = list(int.to_bytes(address, length=2, byteorder='little'))
        command = [0xcc, 0x0f]
        command.extend(address_bytes)
        self.bb.sendpacket(self.get_generic_write_read(command, 0, gap=8))
        self.bb.sendpacket(self.get_generic_write_read(data, 0, gap=0))

        # Read CRC16\:
        self.bb.sendpacket(self.get_generic_write_read([], 2))
        crc = self.bb.recorded_data(2)

        msg = [0x0f]
        msg.extend(address_bytes)
        msg.extend(data)
        expected_crc = DS2431.crc16(msg)

        if crc != expected_crc:
            print('Bad CRC! Expected 0x%x, got 0x%x' % (expected_crc, crc))
        else:
            if verbose: print('Good CRC: 0x%x' % crc)

        # Read FF loop:
        self.bb.sendpacket(self.get_generic_write_read([], 4))
        ffloop = self.bb.recorded_data(4)
        if ffloop != 0xffffffff:
            print('Expected FF loop but got: 0x%x' % ffloop)


    def check_full_scratchpad(self, data, verbose=False):
        self.send_rst_pd(trigger_en=False)
        self.bb.sendpacket(self.get_generic_write_read([0xcc, 0xaa], 0, gap=8))

        # Read TA, E/S:
        self.bb.sendpacket(self.get_generic_write_read([], 3))
        taes = self.bb.recorded_data(3)
        taes_list = [taes & 0xff, (taes >> 8) & 0xff, (taes >> 16) & 0xff]
        if verbose: print('TA, E/S: %s' % (hex(taes)))

        # Read scratchpad:
        self.bb.sendpacket(self.get_generic_write_read([], 8))
        rdata = self.bb.recorded_data(8, False)

        # Read CRC16\:
        self.bb.sendpacket(self.get_generic_write_read([], 2))
        crc = self.bb.recorded_data(2)

        # Calculate CRC:
        msg = [0xaa]
        msg.extend(taes_list)
        msg.extend(rdata)
        expected_crc = DS2431.crc16(msg)
        if crc != expected_crc:
            print('Bad CRC! Expected 0x%x, got 0x%x' % (expected_crc, crc))
        else:
            if verbose: print('Good CRC: 0x%x' % crc)

        # Read FF loop:
        self.bb.sendpacket(self.get_generic_write_read([], 4))
        ffloop = self.bb.recorded_data(4)
        if ffloop != 0xffffffff:
            print('Expected FF loop but got: 0x%x' % ffloop)

        # Check that scratchpad read is what was written:
        for i in range(8):
            assert rdata[i] == data[i], 'Mismatch on read %d: expected 0x%x, read 0x%x' % (i, data[i], rdata[i])

        return taes_list

    def copy_scratchpad(self, taes, trigger_en=True):
        self.send_rst_pd(trigger_en=False)
        self.bb.sendpacket(self.get_generic_write_read([0xcc, 0x55], 0, gap=0), trigger_en=False)
        self.bb.sendpacket(self.get_generic_write_read(taes, 0, gap=0), trigger_en=trigger_en)
        time.sleep(0.1)
        
        # Read AA loop:
        self.bb.sendpacket(self.get_generic_write_read([], 8), trigger_en=False)
        resp = self.bb.recorded_data(8)
        assert resp == 0xaaaaaaaaaaaaaaaa, 'Copy failed! Response: 0x%x' % resp

# From https://gist.github.com/Lauszus/6c787a3bc26fea6e842dfb8296ebd630
    @staticmethod
    def reflect_data(x, width):
        # See: https://stackoverflow.com/a/20918545
        if width == 8:
            x = ((x & 0x55) << 1) | ((x & 0xAA) >> 1)
            x = ((x & 0x33) << 2) | ((x & 0xCC) >> 2)
            x = ((x & 0x0F) << 4) | ((x & 0xF0) >> 4)
        elif width == 16:
            x = ((x & 0x5555) << 1) | ((x & 0xAAAA) >> 1)
            x = ((x & 0x3333) << 2) | ((x & 0xCCCC) >> 2)
            x = ((x & 0x0F0F) << 4) | ((x & 0xF0F0) >> 4)
            x = ((x & 0x00FF) << 8) | ((x & 0xFF00) >> 8)
        elif width == 32:
            x = ((x & 0x55555555) << 1) | ((x & 0xAAAAAAAA) >> 1)
            x = ((x & 0x33333333) << 2) | ((x & 0xCCCCCCCC) >> 2)
            x = ((x & 0x0F0F0F0F) << 4) | ((x & 0xF0F0F0F0) >> 4)
            x = ((x & 0x00FF00FF) << 8) | ((x & 0xFF00FF00) >> 8)
            x = ((x & 0x0000FFFF) << 16) | ((x & 0xFFFF0000) >> 16)
        else:
            raise ValueError('Unsupported width')
        return x

    @staticmethod
    def crc_poly(data, n, poly, crc=0, ref_in=False, ref_out=False, xor_out=0):
        g = 1 << n | poly  # Generator polynomial
        # Loop over the data
        for d in data:
            # Reverse the input byte if the flag is true
            if ref_in:
                d = DS2431.reflect_data(d, 8)
            # XOR the top byte in the CRC with the input byte
            crc ^= d << (n - 8)
            # Loop over all the bits in the byte
            for _ in range(8):
                # Start by shifting the CRC, so we can check for the top bit
                crc <<= 1
                # XOR the CRC if the top bit is 1
                if crc & (1 << n):
                    crc ^= g
        # Reverse the output if the flag is true
        if ref_out:
            crc = DS2431.reflect_data(crc, n)
        # Return the CRC value
        return crc ^ xor_out

# CRC-16/MAXIM
    @staticmethod
    def crc16(msg):
        return DS2431.crc_poly(msg, 16, 0x8005, ref_in=True, ref_out=True, xor_out=0xFFFF)


In [ ]:
target = DS2431(scope.bitbanger)

In [ ]:
scope.clock.clkgen_freq = 20e6
scope.clock.adc_mul = 1
scope.bitbanger.clk_div = 200

`scope.bitbanger` uses the ADC clock and divides it by `scope.bitbanger.clk_div` to drive bits out.

These settings give us a "bit time" of 10 microseconds:

In [ ]:
print('%3.1f microseconds' % (1 / scope.clock.adc_freq * scope.bitbanger.clk_div * 1e6))

## Communicating with the Target:

The first step in communicating with a 1-wire target is to send a reset pulse; the target should respond with a "presense" pulse.

If this runs without errors or warnings, all is good.

If not, check your connections, and check that you have a proper pull-up resistor on the I/O line.

In [ ]:
target.send_rst_pd()

Next, we can read the target's unique ROM code:

In [ ]:
target.read_rom_code()

The DS2431 has memory protection features. If copy protection has been enabled on your device, you won't be able to write to it:

In [ ]:
target.mem_protect_status()

## Reading and Writing Memory

We do 8-byte writes since that's the size of the DS2431's scratchpad, but you can easily build on this to read/write the full memory.

In [ ]:
target.read_mem(address=0, num_bytes=8)

Writing memory requires the following steps:

In [ ]:
target.write_full_scratchpad(address=0, data=range(8))

In [ ]:
taes = target.check_full_scratchpad(data=range(8))

In [ ]:
target.copy_scratchpad(taes=taes)

Let's check that the write worked:

In [ ]:
target.read_mem(address=0, num_bytes=8)

# Power Traces

Let's capture some power traces! We start with the basic reset/presence.

We set our capture to be triggered by `scope.bitbanger`.

At the same time, we'll capture the digital state of the I/O line using `scope.LA`.

`scope.LA` doesn't have a "presamples" feature like `scope.adc`, so we'll manually shift things accordingly.

In [ ]:
scope.trigger.module = 'bitbanger'
scope.gain.db = 10
scope.adc.samples = 40000
scope.adc.presamples = 10000

scope.LA.enabled = True
scope.LA.clk_source = 'pll'
scope.LA.trigger_source = 'falling_tio1'
scope.LA.downsample = 1
scope.LA.capture_group = 'CW 20-pin'
scope.LA.capture_depth = scope.adc.samples

In [ ]:
def capture_trace(sendcommand):
    scope.arm()
    sendcommand()
    ret = scope.capture()
    if ret:
        print("WARNING: Timeout happened during capture")
        return None
    wave = scope.get_last_trace()
    return wave

In [ ]:
scope.LA.arm()
t = capture_trace(lambda: target.send_rst_pd(trigger_bit=0))

In [ ]:
io_data = scope.LA.extract(scope.LA.read_capture_data(), 0)
io_shifted = [1]*scope.adc.presamples
io_shifted.extend(io_data[:scope.adc.samples-scope.adc.presamples])

In [ ]:
import numpy as np
cw.plot(t) * cw.plot(np.asarray(io_shifted)+0.5)

In [ ]:
scope.errors.clear()

There is a lot of interesting stuff happening here!

**First,** remember that Husky's ADC is AC-coupled. This essentially means that it captures *changes* in the input, not the absolute value of the input.

Looking at the red curve (the digital state of the I/O line), the first low pulse is the reset pulse driven by Husky. On the blue curve, we see the target immediately respond and behave differently.

Zooming in on this part of the power trace, you should see a fairly clear periodic pattern with peaks every ~160 samples. This suggests an internal clock of approximately 125 KHz.

**Next,** we find a pair of spikes a bit *before* the end of the reset pulse. Let's look at exactly how this reset pulse is driven out. Inspecting `target.send_rst_pd`, we see that it calls `target.get_rst_pd(rst=56, wait=6, check=2)`. That method returns a `BitBangerPacket` object; let's look at it:

In [ ]:
target.send_rst_pd??

In [ ]:
p = target.get_rst_pd(rst=56, wait=6, check=2)
print(p)

Here we find that `p.pattern_hiz` goes high on bit 56:

In [ ]:
p.pattern_hiz[55:]

This means that on bit 56, we tristate the I/O line (so that the target may drive it).

If you do the math, you'll find that this coincides with the spikes on the power trace.

**Finally**, we see the target's presence response, and a short time after that, the power signature changes again and returns to the idle state that was seen prior to the reset pulse.

What if we collect and average a large number of traces?

In [ ]:
import numpy as np
from tqdm.notebook import tnrange

avgtrace = np.zeros(scope.adc.samples)
for i in tnrange(1000):
    trace = capture_trace(lambda: target.send_rst_pd(trigger_bit=1))
    avgtrace += trace

In [ ]:
cw.plot(avgtrace)

Two things jump out:
1. Thanks to the hardware bit-banging's consistent timing, we have very well synchronized traces! None of the edges - whether driven by Husky or by the target - have any jitter.
2. The periodic pulses that were suggestive of a 125 KHz clock in the single power trace are less apparent, although we do see them between samples 21500 and 31000.

It is not surprising for the target clock to disappear with multiple averaged traces, since that clock has no fixed relationship with the ADC sampling clock (the trace averaging "smooshes" out the clock).

It *is* interesting that in spite of that, the presence response rising and falling edges are rock steady, and that a target clocks does remain synchronized to our reset pulse at certain times.

This very basic example illustrates one of the useful aspects of `scope.bitbanger`: while you could use software bit-banging to drive the reset pulse, and easily trigger the capture from the I/O line, the width of the reset pulse would be highly variable (and not finely controllable); you would not be able to stack up multiple traces as we did here and have all the edges line up nicely.

# Memory Write Activity

To close it off, let's try to gain insights into when the target does its EEPROM writes.

The datasheet tells us that the maximum programming time is 10ms.

It also tells us that the target draws more current when writing EEPROM: 0.8mA, up from 0.05 - 6.7uA.

What can power traces tell us?

First we'll customize `target.copy_scratchpad` so that we can specify when we wish to trigger our power trace capture:

In [ ]:
target.copy_scratchpad??

In [ ]:
def copy_scratchpad_trig(target, taes, trigger_bit=None):
    target.send_rst_pd(trigger_en=False)
    packet = target.get_generic_write_read([0xcc, 0x55], 0, gap=0)
    packet.extend(target.get_generic_write_read(taes, 0, gap=0))
    if trigger_bit != None:
        packet.trig_bits[trigger_bit] = 1
    target.bb.sendpacket(packet)
    time.sleep(0.1)
    
    # Read AA loop:
    target.bb.sendpacket(target.get_generic_write_read([], 8))
    resp = target.bb.recorded_data(8)
    assert resp == 0xaaaaaaaaaaaaaaaa, 'Copy failed! Response: 0x%x' % resp

def write_mem_trig(addr, wdata, trigger_bit):
    assert len(wdata) == 8
    target.write_full_scratchpad(addr, wdata)
    taes = target.check_full_scratchpad(wdata)
    copy_scratchpad_trig(target, taes, trigger_bit)

Let's take it for a spin:

In [ ]:
write_mem_trig(addr=0, wdata=[0xff]*8, trigger_bit=0)
assert target.read_mem(0, 8) == [0xff]*8

Now we capture a bunch of traces. We'll use the `scope.adc.decimate` feature so that the number of samples needed to see what's interesting doesn't exceed Husky's capacity. (If you have a Husky Plus, you can increase `samples` to 300000 and leave `decimate` at 1.)

In [ ]:
scope.adc.samples = 100000
scope.adc.presamples = 0
scope.adc.decimate = 4

In [ ]:
import numpy as np
from tqdm.notebook import tnrange

avgtrace = np.zeros(scope.adc.samples)
for i in tnrange(50):
    trace = capture_trace(lambda: write_mem_trig(0, range(8), -1))
    avgtrace += trace

In [ ]:
cw.plot(avgtrace)

We triggered the capture on the last bit of the copy scratchpad command; we can see that bit ending around sample 1650.

Then we find a pair of large peaks around sample 22300 and another pair around sample 43000.

These are around 4 and 8ms after the end of the copy scratchpad command: they could be pinpointing where the EEPROM write starts and finishes.

## Conclusion
While we haven't done any attacks or retrieved any secrets, we've shown that despite lacking access to a 1-wire target's internal clock or power, we can find out a lot about what it's doing.

Hopefully this serves as a useful springboard for working with 1-wire targets.